In [5]:
import pandas as pd

df = pd.read_csv("data_clean.csv")
df.head()

,date,temp_min,temp_max,temp_mean
0,1996-04-02,-4.0,2.0,-1.00
1,1996-04-03,1.0,4.4,2.70
2,1996-04-04,2.0,4.0,3.00
3,1996-04-05,0.8,3.4,2.10
4,1996-04-06,0.2,9.1,4.65


In [6]:
df["date"] = pd.to_datetime(df["date"])
df['day_of_year'] = pd.to_datetime(df['date']).dt.day_of_year
df['month']       = pd.to_datetime(df['date']).dt.month
df['year']        = pd.to_datetime(df['date']).dt.year

In [7]:
print(df["date"].dtype)

datetime64[us]


In [8]:
from sklearn.model_selection import train_test_split

df['date'] = df['date'].map(lambda x: x.toordinal())
X = df[["date", 'day_of_year', 'month', 'year']]
y = df['temp_mean']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = GradientBoostingRegressor()
model.fit(X_train, y_train)

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
from sklearn.metrics import mean_squared_error

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)

mse

5.433215612204535

In [ ]:
import pandas as pd

user_date = pd.to_datetime("2016-03-01")

user_input = pd.DataFrame({
    'date':        [user_date.toordinal()],
    'day_of_year': [user_date.day_of_year],
    'month':       [user_date.month],
    'year':        [user_date.year]
})

predicted_temp = model.predict(user_input)
print(f"Predicted temp_mean: {predicted_temp[0]:.1f} °C")

Predicted temp_mean: 2.7 °C


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

model_low  = GradientBoostingRegressor(loss='quantile', alpha=0.1)  # 10th percentile
model_mid  = GradientBoostingRegressor(loss='quantile', alpha=0.5)  # median
model_high = GradientBoostingRegressor(loss='quantile', alpha=0.9)  # 90th percentile

model_low.fit(X_train, y_train)
model_mid.fit(X_train, y_train)
model_high.fit(X_train, y_train)

# Predict
low  = model_low.predict(user_input)[0]
mid  = model_mid.predict(user_input)[0]
high = model_high.predict(user_input)[0]

print(f"Predicted: {mid:.1f} °C  (range: {low:.1f} – {high:.1f} °C)")
print(f"+- {round((high - low) / 2)}")

In [ ]:
import joblib
from pathlib import Path

models_path = Path("models")
models_path.mkdir(exist_ok=True)

# Save
models = {
    'low':  model_low,
    'mid':  model_mid,
    'high': model_high
}
joblib.dump(models, models_path / 'temperature_models.pkl')

In [ ]:
models = joblib.load(models_path / 'temperature_models.pkl')
predicted = models['mid'].predict(user_input)
predicted[0]

In [ ]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500, 
    learning_rate=0.05,   
    max_depth=6,           
    subsample=0.8,         
    colsample_bytree=0.8,  
    random_state=42
)

model